In [23]:
import os
import json
import getpass
from dotenv import load_dotenv

# .env 파일 로드 (부모 디렉토리 또는 루트 경로)
dotenv_path = os.path.join("..", ".env")
load_dotenv(dotenv_path)

# 경로 설정
DATA_DIR = os.path.join("..", "data", "proceed")
RAG_DATA_PATH = os.path.join(DATA_DIR, "kf_area_rag_data.json")
OUTPUT_DIR = os.path.join(DATA_DIR, "scenarios")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"RAG 데이터 경로: {os.path.abspath(RAG_DATA_PATH)}")
print(f"결과 저장 경로: {os.path.abspath(OUTPUT_DIR)}")


RAG 데이터 경로: /Users/ahramkim/Documents/GitHub/k-heroes/backend/data/proceed/kf_area_rag_data.json
결과 저장 경로: /Users/ahramkim/Documents/GitHub/k-heroes/backend/data/proceed/scenarios


## 1. RAG 마스터 데이터 로드

In [24]:
# RAG 데이터 로드
if os.path.exists(RAG_DATA_PATH):
    with open(RAG_DATA_PATH, "r", encoding="utf-8") as f:
        rag_data = json.load(f)
    print(f"[SUCCEED] 총 {len(rag_data)}개의 RAG 단서 데이터를 메모리에 로드 완료.")
else:
    print(f"[ERROR] RAG 데이터 파일을 찾을 수 없습니다. 전처리 확인해주세요: {RAG_DATA_PATH}")


[SUCCEED] 총 24434개의 RAG 단서 데이터를 메모리에 로드 완료.


## 2. 대상 인물 선택 및 관련 RAG 데이터 필터링

In [25]:
# 1. 대상 인물 선택
SELECTED_CHARACTER = "이순신"

# 2. RAG 데이터 필터링
retrieved_clues = []
for item in rag_data:
    metadata = item.get("metadata", {})
    related_person = metadata.get("related_person") or ""
    text = item.get("text") or ""
    
    if SELECTED_CHARACTER in related_person or SELECTED_CHARACTER in text:
        retrieved_clues.append(item)

print(f"==================================================")
print(f" [{SELECTED_CHARACTER}] 관련 RAG 데이터 검색 결과")
print(f" - 매칭된 단서 수: 총 {len(retrieved_clues)}개")
print(f"==================================================")

# 검색된 단서의 일부 샘플 확인 (최대 3개)
for i, clue in enumerate(retrieved_clues[:3]):
    print(f" 단서 {i+1} [출처 ID: {clue['metadata'].get('source_id')} / 지역: {clue['metadata'].get('region_sido')} {clue['metadata'].get('region_sigungu')}]")
    print(f" > {clue['text'][:120]}...")
    print("-" * 50)


 [이순신] 관련 RAG 데이터 검색 결과
 - 매칭된 단서 수: 총 110개
 단서 1 [출처 ID: 908 / 지역: 충청남도 아산시]
 > [시뮬레이터 단서] 이 문서는 '충청남도' 지역의 지역 이야기와 지역 인물에 관한 역사 기록입니다. 자료의 명칭은 '1등 공신으로 책훈하는 이순신 선무공신교서'이며, 관련된 역사적 사건 및 핵심 요약은 다음과 같습니...
--------------------------------------------------
 단서 2 [출처 ID: 1390 / 지역: 전라남도 해남군]
 > [시뮬레이터 단서] 이 문서는 '전라남도' 지역의 지역 이야기와 지역 인물에 관한 역사 기록입니다. 자료의 명칭은 '명량해전의 승리를 기리는 전남 해남 명량대첩비'이며, 관련된 역사적 사건 및 핵심 요약은 다음과 같...
--------------------------------------------------
 단서 3 [출처 ID: 2989 / 지역: 경상남도 통영시]
 > [시뮬레이터 단서] 이 문서는 '경상남도' 지역의 지역 이야기와 지역 인물에 관한 역사 기록입니다. 자료의 명칭은 '동양 최초의 바다 밑 터널, 통영 해저터널'이며, 관련된 역사적 사건 및 핵심 요약은 다음과 같습니...
--------------------------------------------------


## 3. Gemini API 클라이언트 설정 및 인증

In [26]:
# Gemini API Key 가져오기
gemini_api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

if not gemini_api_key:
    print("Gemini API Key가 환경 변수에 등록되어 있지 않습니다.")
    gemini_api_key = getpass.getpass("Gemini/Google API Key를 입력하세요: ")
    os.environ["GEMINI_API_KEY"] = gemini_api_key
else:
    print("환경 변수에서 Gemini API Key를 로드 완료!")


환경 변수에서 Gemini API Key를 로드 완료!


## 4. [1단계] 인물 설명 카드 및 시나리오 분기 뼈대 생성

In [27]:
# RAG 컨텍스트 문자열 합성
context_str = ""
for i, clue in enumerate(retrieved_clues):
    context_str += f"[단서 {i+1}]\n{clue['text']}\n\n"

# 1단계 프롬프트 작성
part1_prompt = f"""# 역할
너는 역사 콘텐츠 기반의 A/B 선택형 시뮬레이션 게임(서비스명: K-Heroes)의 시나리오 설계자야.
초등학생 및 외국인(역사 입문자) 타겟에 맞춰 아주 쉽고 직관적인 용어를 사용하며, 모바일 UI 화면에 최적화된 호흡이 짧은 텍스트를 생성해야 해.

[1단계 목표]
대상 인물인 '{SELECTED_CHARACTER}'에 대한 '1. 인물 설명 카드'와 '2. 시나리오 분기(STEP 1 ~ STEP 3)'를 생성하는 것이야.

[RAG 역사 백그라운드 데이터 컨텍스트]
{context_str}

# 제작 가이드라인 (1단계)
1. 역사적 사실 기반: 제공된 RAG 데이터에 나타난 역사적 사실(사건, 활동 지역, 업적 등)을 바탕으로 인물 설명과 시나리오 상황을 설정하세요.
2. 언어 수준: '순국', '타전', '수뇌부', '반정', '망명' 같은 어려운 한자어를 철저히 배제하고, 초등학생도 5초 만에 읽고 뜻을 바로 이해할 수 있도록 명확하고 쉬운 순우리말이나 직관적인 표현으로 작성해 주세요.
3. 맞춤형 시스템 지표(스탯) 설정:
   - 인물 배경에 맞게 3가지 시스템 지표(스탯)를 창작하세요. (예: [군사들의 사기 🔥 / 거북선 내구도 🛡️ / 전쟁 승리 확률 ⚔️])
   - 초기 수치는 각각 50 / 50 / 50%로 고정하고, 선택지 결과에 따라 이 3가지 지표가 증감하게 설정합니다.
4. 강점 특기 스탯 3개 설정:
   - 인물의 강점을 나타내는 특기 스탯 3개를 선정해 별점 표시(★, ☆)와 `[X/5]` 형식으로 부여하고, 쉬운 한 줄 설명을 적어주세요.
5. MBTI 및 스토리 인트로 작성:
   - 인물의 성향에 어울리는 MBTI(4글자)와 별명, 그리고 인트로 대사와 다짐 문장을 작성하세요.
6. 시나리오 분기 구조:
   - 총 3단계(STEP 1~3)로 진행되며, 각 STEP마다 메인 상황, 질문 형태의 토글 버튼 상세 설명, 그리고 유저 선택지 A, B를 생성합니다.
   - 각 선택지 뒤에는 3가지 시스템 지표의 증감 수치를 반드시 기재하세요. (예: `[스탯1 +10 / 스탯2 -5 / 스탯3 +15%]`)
   - 8가지 분기 경로(AAA ~ BBB) 중 딱 1개의 경로만 실제 역사(True)와 100% 일치하고, 나머지 7개는 개연성 있는 가상(Alternative) 시나리오여야 합니다. 정답 경로(True)는 무작위 조합(예: A-B-A 등)으로 설정하세요.

# 출력 포맷 (1단계)
### 1. 인물 설명
- **이름:** 
- **시대:** (예: 조선 시대 1545~1598)
- **상황:** (초등·외국인 맞춤형 시대 상황 한줄 요약)
- **한줄 요약:** (히어로물 느낌의 직관적인 수식어)
- **강점:**
    - **[인물 맞춤형 특기 스탯 1]:** ★★★★★ `[X/5]` (쉬운 설명)
    - **[인물 맞춤형 특기 스탯 2]:** ★★★★★ `[X/5]` (쉬운 설명)
    - **[인물 맞춤형 특기 스탯 3]:** ★★★★★ `[X/5]` (쉬운 설명)

<aside>
🦸🏻
**[MBTI]**
**[MBTI 4글자] (인물 특징에 맞는 별명)**
- **I/E:** 한 줄 설명
- **N/S:** 한 줄 설명
- **F/T:** 한 줄 설명
- **P/J:** 한 줄 설명
</aside>

<aside>
🦸🏻
**[스토리 인트로]** **\"[인물의 유명한 대사나 다짐 한 줄]\"**
(인물이 처했던 상황과 이 사건을 결심하게 된 계기를 설명하는 쉬운 문장 2~3줄)
이제 당신이 [인물명]이 되어 역사의 거대한 소용돌이 속에서 운명을 바꿀 중요한 선택들을 내려야 합니다. 준비되셨나요?
</aside>

### 2. 시나리오

#### STEP 1. [단계 제목]
- **[메인 상황]** (모바일용 2줄 요약 상황)
- **[토글 버튼] 💡 [메인 상황과 토글 내용을 대표하는 짧고 핵심적인 질문]? (클릭 시 노출)**
  > (배경지식이 없는 유저를 위한 쉬운 역사 맥락 설명)
- **[유저 선택지] (A/B)**
  - **[선택 A - 실제 역사 또는 가상 분기]** 선택지 내용 `[지표 3개의 증감 수치]`
  - **[선택 B - 실제 역사 또는 가상 분기]** 선택지 내용 `[지표 3개의 증감 수치]`

#### STEP 2. [단계 제목]
- **[메인 상황]**
- **[토글 버튼] 💡 [핵심 질문]?**
  > (역사적 허점/이유/선택의 근거 설명)
- **[유저 선택지]**
  - **[선택 A]** 내용 `[지표 증감 수치]`
  - **[선택 B]** 내용 `[지표 증감 수치]`

#### STEP 3. [단계 제목]
- **[메인 상황]**
- **[토글 버튼] 💡 [핵심 질문]?**
  > (역사적 이유 설명)
- **[유저 선택지]**
  - **[선택 A]** 내용 `[지표 증감 수치]`
  - **[선택 B]** 내용 `[지표 증감 수치]`
"""

print("[1단계 프롬프트 구성 완료]")


[1단계 프롬프트 구성 완료]


In [28]:
import google.generativeai as genai

print("[1단계] Gemini 호출 시작...")
genai.configure(api_key=gemini_api_key)
model = genai.GenerativeModel("gemini-2.5-flash")
response = model.generate_content(part1_prompt)
part1_result = response.text

print("\n✔ [1단계 완료] 생성 결과:")
print(part1_result)


[1단계] Gemini 호출 시작...

✔ [1단계 완료] 생성 결과:
### 1. 인물 설명
- **이름:** 이순신
- **시대:** 조선 시대 1545~1598 (임진왜란)
- **상황:** 일본이 쳐들어와 나라가 위험에 빠진 조선 시대, 바다를 지켜야 했던 상황.
- **한줄 요약:** 흔들림 없는 용기로 바다를 지킨 영웅!
- **강점:**
    - **지략 (Strategy):** ★★★★★ `[5/5]` (적의 움직임을 꿰뚫어 보는 능력)
    - **통솔력 (Leadership):** ★★★★★ `[5/5]` (수군을 하나로 이끌어 승리하는 능력)
    - **애민정신 (Love for People):** ★★★★★ `[5/5]` (나라와 백성을 지키려는 따뜻한 마음)

<aside>
🦸🏻
**[MBTI] INTJ (바다의 영웅)**
- **I (내향):** 혼자 생각하고 계획하며 깊이 있는 고민을 즐겨요.
- **N (직관):** 눈에 보이는 것보다 숨겨진 가능성과 큰 그림을 잘 봐요.
- **T (사고):** 감정보다는 논리와 규칙으로 문제를 해결하려고 해요.
- **J (판단):** 계획을 세우고 실행하며, 결정을 내리는 것을 중요하게 생각해요.
</aside>

<aside>
🦸🏻
**[스토리 인트로]** **"나의 죽음을 적에게 알리지 마라!"**
일본이 조선을 쳐들어와 온 나라가 불타고 백성들이 고통받고 있습니다. 이순신 장군은 나라를 지키기 위해 두려움 없이 바다로 나섭니다.
이제 당신이 이순신 장군이 되어 역사의 거대한 소용돌이 속에서 운명을 바꿀 중요한 선택들을 내려야 합니다. 준비되셨나요?
</aside>

### 2. 시나리오

#### STEP 1. 절망 속 마지막 기회
- **[메인 상황]**
  이순신 장군은 모든 관직을 잃고 평범한 병사로 싸우고 있습니다.
  하지만 일본군에게 크게 진 조선 수군은 다시 장군을 찾습니다.
- **[토글 버튼] 💡 다시 싸우러 가야 할까요? (클릭 시 노출)**
  > 선조 임금은

## 5. [2단계] 8가지 엔딩 분기(RESULT 1~8) 상세 생성

In [29]:
# 2단계 프롬프트 작성
part2_prompt = f"""# 역할
너는 역사 콘텐츠 기반의 A/B 선택형 시뮬레이션 게임(서비스명: K-Heroes)의 시나리오 엔딩 설계자야.

[2단계 목표]
1단계에서 설계된 시나리오 뼈대를 바탕으로, 8가지 모든 조합 경로(AAA ~ BBB)에 따른 '3. 최종 결과' 마크다운 엔딩 데이터를 작성해야 해. 

[1단계 시나리오 정보]
{part1_result}

[RAG 역사 백그라운드 데이터 컨텍스트]
{context_str}

# 제작 가이드라인 (2단계 - 필수 준수)
1. 엔딩 생략 금지: RESULT 1(AAA)부터 RESULT 8(BBB)까지 8개의 모든 조합을 절대로 요약하거나 생략하지 말고 하나도 빠짐없이 전부 작성해 줘. (텍스트가 잘리지 않도록 핵심만 명확하고 짧게 호흡을 유지하며 작성)
2. 능력치 결과 밸런스: STEP 1~3을 거치며 누적 합산된 3가지 지표의 최종 수치를 각 RESULT마다 논리적으로 매칭해서 표기해 줘 (기본치 50에서 시작하여 증감 합산).
3. 실제 역사와 비교:
   - 이 8가지 경로 중 딱 1개 경로만 실제 역사와 100% 일치합니다. 1단계의 실제 역사 선택지 조합을 토대로 True 엔딩을 기재해 주세요.
   - 실제 역사 경로(True)라면 축하 문구와 역사적 의의를, 가상 시나리오라면 유저의 선택이 실제 역사와 어떻게 다르고 어떤 결과를 낳았을지 당시 시대 배경을 고려한 개연성 있는 대체 역사(Alternative History) 팩트체크로 서술해 줘 (마법이나 비현실적인 판타지는 금지).
4. 결과 스토리 요약:
   - '📖 내가 만든 이야기 (Story)' 부분은 효과음이나 대사로 시작하는 강렬한 헤드라인 1줄과 극적인 서사 2줄의 짧은 문장으로 압축해 줘.
   - '📌 결과 요약 (Summary)'은 이 선택으로 벌어진 역사적/정치적 결과를 쉬운 단어의 불릿 포인트 3개로 요약해 줘.

# 출력 포맷 (이 형식으로 RESULT 1(AAA)부터 RESULT 8(BBB)까지 8번 반복 출력)
### 3. 최종 결과

#### RESULT X. [선택 조합] [🅾️/❎ 이모지 3개](예: A-A-A 🅾️ ❎ 🅾️ / A-A-B 🅾️ ❎ ❎ 등)
**[상단 타이틀 및 캐릭터 연출]**
- **엔딩 타이틀:** 🔴/🔵 엔딩 제목 (★실제 역사와 100% 일치★ 또는 가상 시나리오 표시)
- **최종 능력치 결과:** (1단계에서 설정된 지표 3개의 최종 합산 수치 표기)

**[실제 역사적 사실과 비교]**
- **💡 이 엔딩은 실제 역사와 어떤 차이가 있을까요?**
- (역사적 팩트체크 및 역사적 의의 서술)

**[결과 스토리 및 요약]**
- **📖 내가 만든 이야기 (Story)**
  - **\"[효과음이나 대사 중심의 강렬한 헤드라인]\"**
  - (결과 묘사 짧은 서사 2줄)
- **📌 결과 요약 (Summary)**
  - (결과 요약 불릿 포인트 3개)

#### 추천 방문지
- 💡 {SELECTED_CHARACTER} 장군을 좀 더 알아보고 싶으세요?
- (관련 기념관, 생가, 박물관이나 유적지 등 2~3곳 추천)

#### Floating Button
- **공유하기:** \"내가 만든 역사 엔딩을 친구에게 공유해보세요! 🔗\"
- **다음 인물 체험하기**
"""

print("[2단계 프롬프트 구성 완료]")


[2단계 프롬프트 구성 완료]


In [30]:
print("[2단계] Gemini 호출 시작...")
response2 = model.generate_content(part2_prompt)
part2_result = response2.text

print("\n✔ [2단계 완료] 생성 결과:")
print(part2_result)


[2단계] Gemini 호출 시작...

✔ [2단계 완료] 생성 결과:
### 3. 최종 결과

#### RESULT 1. A-A-A 🅾️ 🅾️ ❎
**[상단 타이틀 및 캐릭터 연출]**
- **엔딩 타이틀:** 🔵 피로 얼룩진 승리 (가상 시나리오)
- **최종 능력치 결과:** 군사들의 사기 85 / 거북선 내구도 45 / 전쟁 승리 확률 65%

**[실제 역사적 사실과 비교]**
- **💡 이 엔딩은 실제 역사와 어떤 차이가 있을까요?**
  실제 역사에서 이순신 장군은 명량 해전 후 한산도 대첩에서 '학익진'이라는 기발한 전략으로 대승을 거두었습니다. 하지만 이 엔딩에서는 정면 승부를 선택하여 초기 사기는 높았으나, 전력 손실이 커져 귀중한 수군 전력을 크게 잃게 되었습니다. 비록 승리는 거두었으나, 이후 전쟁의 양상이 조선에 더 불리하게 흘러갔을 것입니다. 지략을 통한 효율적인 승리 대신, 무모한 용기만이 앞섰던 결과입니다.

**[결과 스토리 및 요약]**
- **📖 내가 만든 이야기 (Story)**
  - **"함성! 그러나 전선 곳곳에서 피가 솟구친다!"**
  - 무모한 정면 돌격은 적을 물리쳤으나, 아군 또한 막대한 피해를 입었다. 승리의 환호 속에서도, 남은 병사들의 얼굴에는 깊은 상처와 불안감이 드리웠다.
- **📌 결과 요약 (Summary)**
  - 수많은 희생을 치른 대가로 전쟁의 승리 확률은 낮아졌습니다.
  - 조선 수군은 중요한 전력을 잃어 향후 해전에서 큰 어려움을 겪게 됩니다.
  - 이순신 장군의 리더십은 인정받았으나, 백성들의 피로도는 극심해집니다.

#### 추천 방문지
- 💡 이순신 장군을 좀 더 알아보고 싶으세요?
- 충청남도 아산시 '현충사' (이순신 장군의 충의를 기리는 사당)
- 전라남도 해남군 '명량대첩비' (명량 해전의 승리를 기념하는 비)
- 경상남도 통영시 '통영 충렬사' (이순신 장군의 위패를 모신 사당)

#### Floating Button
- **공유하기:** "내가 만든 역사 엔딩을 친구

## 6. 최종 시나리오 데이터 통합 및 마크다운 저장

In [ ]:
# # 1단계와 2단계 결과물을 자연스럽게 하나로 통합
# full_scenario_content = f"""# 시뮬레이션 가이드

# ## ◾️[{SELECTED_CHARACTER}] 시뮬레이션 데이터

# {part1_result}

# {part2_result}
# """

# output_file_name = f"03_{SELECTED_CHARACTER}_full_scenario.md"
# output_file_path = os.path.join(OUTPUT_DIR, output_file_name)

# with open(output_file_path, "w", encoding="utf-8") as f:
#     f.write(full_scenario_content)

# print(f"✔ [저장 완료] 2단계 시나리오 빌드가 성공적으로 완료되었습니다!")
# print(f" -> 최종 마크다운 파일: {os.path.abspath(output_file_path)}")
